In [1]:
import os
import sys
from datetime import timedelta

import pandas as pd
import geopandas as gpd
import numpy as np

from projects.scotland.scripts import build_windows

# Get the path to your active conda environment
conda_prefix = sys.prefix
# Set the PROJ_LIB variable to the correct share/proj directory
os.environ['PROJ_LIB'] = os.path.join(conda_prefix, 'share', 'proj')

In [ ]:
phs_test = pd.read_csv(
    "../data/raw/scotland/phs/confidential/PHS_CovidTesting_2023-02-22.csv",
    parse_dates=["date_ecoss_specimen"], date_format="%Y%m%d"
)
phs_test.rename(
    columns={
        "date_ecoss_specimen": "collection_date",
        "PatientID": "patient_id",
        "datazone2011": "datazone",
    },
    inplace=True
)

phs_test.sort_values(by=["collection_date"], inplace=True)
phs_test.drop_duplicates(subset=["patient_id", "specimen_id", "datazone"], keep="first", inplace=True)


phs_test.sort_values(by=["collection_date"], inplace=True)
phs_test.drop_duplicates(subset=["patient_id", "specimen_id", "datazone"], keep="first", inplace=True)

scot_testing = phs_test.groupby(["collection_date", "datazone"]).size().reset_index(name='dz_total_tests')
positive_tests = phs_test[phs_test["test_result"] == "POSITIVE"].groupby(["collection_date", "datazone"]).size().reset_index(name='dz_positive_tests')
negative_tests = phs_test[phs_test['test_result'] == 'NEGATIVE'].groupby(["collection_date", "datazone"]).size().reset_index(name='dz_negative_tests')

testing = (scot_testing
           .merge(positive_tests, on=["collection_date", "datazone"], how='left')
           .merge(negative_tests, on=["collection_date", "datazone"], how='left'))

testing.fillna(0, inplace=True)

assert len(testing) == len(scot_testing)
assert testing.notna().all().all()

testing.to_parquet("../data/processed/scotland_testing.parquet")


In [ ]:
simd = pd.read_csv("../data/raw/scotland/datazone-characteristics/2020v2_simd.csv")

simd.rename(
    columns={
        "DZ": "datazone",
        "Population": "dz_population",
        "Working_Age_Population": "dz_working_age_population",
        "SIMD2020v2_Rank": "dz_simd_rank",
        "SIMD2020v2_Quintile": "dz_simd_quintile",
        "SIMD2020v2_Decile": "dz_simd_decile",
        "SIMD2020v2_Vigintile": "dz_simd_vigintile",
        "SIMD2020v2_Income_Domain_Rank": "dz_simd_income_rank",
        "SIMD2020_Employment_Domain_Rank": "dz_simd_employment_rank",
        "SIMD2020_Education_Domain_Rank": "dz_simd_education_rank",
        "SIMD2020_Health_Domain_Rank": "dz_simd_health_rank",
        "SIMD2020_Access_Domain_Rank": "dz_simd_access_rank",
        "SIMD2020_Crime_Domain_Rank": "dz_simd_crime_rank",
        "SIMD2020_Housing_Domain_Rank": "dz_simd_housing_rank",

    },
    inplace=True
)

simd = simd[[
    "datazone",
    "dz_population",
    "dz_working_age_population",
    "dz_simd_rank",
    "dz_simd_quintile",
    "dz_simd_decile",
    "dz_simd_vigintile",
    "dz_simd_income_rank",
    "dz_simd_employment_rank",
    "dz_simd_education_rank",
    "dz_simd_health_rank",
    "dz_simd_access_rank",
    "dz_simd_crime_rank",
    "dz_simd_housing_rank"]]

assert simd.notna().all().all()

simd.to_parquet("../data/processed/scotland_datazone_simd_data.parquet")

In [ ]:
simd_mod = simd.set_index("datazone")

scot_geography = gpd.read_file("../data/raw/scotland/datazone-characteristics/sg_datazone_bdry_2011.shp").set_index("DataZone")
scot_geography.index.name = "datazone"

# Centroids and planar coords
scot_geography["dz_centroid"] = scot_geography.geometry.centroid
scot_geography["dz_x"] = scot_geography["dz_centroid"].x
scot_geography["dz_y"] = scot_geography["dz_centroid"].y

scot_geography = (scot_geography[['Name', 'geometry', 'dz_centroid', 'dz_x', 'dz_y']]
                  .merge(simd_mod, how="left", left_index=True, right_index=True)
                  )

scot_geography["dz_geometry"] = scot_geography["geometry"]

scot_geography.to_parquet("../data/processed/scotland_geography.parquet")

In [ ]:
scot_geography.columns

In [ ]:
vaccination = pd.read_csv(
    "../data/raw/scotland/phs/confidential/PHS_Vaccinations_2023-02-22.csv",
    low_memory=False,
)
vaccination["vacc_occurence_time"] = pd.to_datetime(vaccination["vacc_occurence_time"], format="%Y%m%d", errors='coerce')

vaccination.dropna(subset=["vacc_occurence_time"], inplace=True)
vaccination.dropna(subset=["age_band"], inplace=True)
vaccination.rename(columns={
    "PatientID": "patient_id",
    "vacc_occurence_time": "vaccination_date",
    "datazone2011": "datazone"
}, inplace=True)

assert vaccination[["vaccination_date", "age_band", "patient_id"]].notna().all().all()

def band_to_midpoints(bands: pd.Series) -> pd.Series:
    """
    Convert age-band strings into numeric midpoints.
    Supports forms like "0-4", "5-9", ..., "75+" (treated as [75,80) -> midpoint 77.5).
    Returns NaN if parsing fails.
    """
    s = bands.astype('string')
    lower = s.str.extract(r'(\d+)')[0].astype(float)
    upper = s.str.extract(r'-(\d+)')[0].astype(float)
    open_ended = s.str.endswith('+').fillna(False)
    upper = np.where(open_ended, lower + 5.0, upper)
    with np.errstate(invalid='ignore'):
        mid = (lower + upper) / 2.0
    return pd.to_numeric(pd.Series(mid, index=bands.index), errors='coerce')

vaccination["age_midpoint"] = band_to_midpoints(vaccination["age_band"])

dz_vaccination = (
    vaccination
    .groupby(["vaccination_date", "datazone"])
    .agg(
        dz_total_vaccinated=("patient_id", "nunique"),
        dz_mean_vacc_age=("age_midpoint", "mean"),
        dz_median_vacc_age=("age_midpoint", "median"),
        dz_mean_vdose_number=("vacc_dose_number", "mean"),
        dz_median_vdose_number=("vacc_dose_number", "median"),
    )
    .reset_index()
)

dz_vaccination.to_parquet("../data/processed/scotland_datazone_vaccinations.parquet")

In [ ]:
nextclade_res = pd.read_table(
    "../data/raw/scotland/cog-uk/cog_all_scotland_nextclade.tsv",
    low_memory=False
)

assert any(~nextclade_res["seqName"].duplicated(keep=False))
nextclade_res["seq_id"] = nextclade_res["seqName"].str.split('/').apply(lambda x: x[1])
nextclade_res.set_index("seq_id", inplace=True)

scot_metadata = pd.read_csv(
    "../data/raw/scotland/phs/confidential/AnnaSequencedCases.csv",
    parse_dates=["Collection_Date"]
)

scot_metadata.rename(
    columns={
        "Collection_Date": "collection_date",
        "subject_sex": "sex",
        "SequenceID": "seq_id",
        "PatientID": "patient_id",
        "datazone2011": "datazone"
    },
    inplace=True
)


assert set(scot_metadata["seq_id"]).issubset(set(nextclade_res.index))
scot_metadata["sequence_id"] = list(nextclade_res.loc[list(scot_metadata["seq_id"]), "seqName"])
scot_metadata["clade"] = list(nextclade_res.loc[list(scot_metadata["seq_id"]), "clade"])
scot_metadata["who_voc"] = list(nextclade_res.loc[list(scot_metadata["seq_id"]), "clade_who"])
scot_metadata["pango_lineage"] = list(nextclade_res.loc[list(scot_metadata["seq_id"]), "Nextclade_pango"])
scot_metadata["nextclade_qc"] = list(nextclade_res.loc[list(scot_metadata["seq_id"]), "qc.overallStatus"])

scot_metadata.sort_values("collection_date", inplace=True)

scot_metadata.drop_duplicates(["specimen_id"], keep="first", inplace=True)

scot_metadata = scot_metadata.merge(
    scot_geography[['dz_geometry', 'dz_centroid', 'dz_x', 'dz_y']],
    left_on="datazone", right_index=True)

scot_metadata["age_midpoint"] = band_to_midpoints(scot_metadata["age_band"])

scot_metadata = scot_metadata[[
    "datazone",
    'dz_geometry',
    'dz_centroid',
    'dz_x',
    'dz_y',
    "collection_date",
    "patient_id",
    "sex",
    "age_band",
    "age_midpoint",
    "sequence_id",
    "clade",
    "who_voc",
    "pango_lineage",
    "nextclade_qc"]].copy()



required_cols = [
    "datazone",
    "collection_date",
    "patient_id",
    "sex",
    "age_band",
    "sequence_id",
    "clade",
    "pango_lineage",
    "nextclade_qc"
]

# Drop rows missing any of the required columns
scot_metadata.dropna(subset=required_cols, inplace=True)

assert scot_metadata[required_cols].notna().all().all()

# Merge metadata with vaccinations
seq_vacc = (
    scot_metadata
    .merge(
        vaccination[["patient_id", "vaccination_date", "vacc_dose_number"]],
        on="patient_id",
        how="left"
    )
)

# Keep only vaccinations that happened before or on collection date
seq_vacc = seq_vacc[seq_vacc["vaccination_date"] <= seq_vacc["collection_date"]]

# Pick the latest vaccination per patient (if any exist before sequencing)
latest_vacc = (
    seq_vacc.sort_values(["patient_id", "vaccination_date"])
    .groupby("patient_id")
    .tail(1)
)

# Re-merge with metadata to ensure all patients are kept
scot_metadata = (
    scot_metadata
    .merge(
        latest_vacc[["patient_id", "vaccination_date", "vacc_dose_number"]],
        on="patient_id",
        how="left"
    )
)

scot_metadata["is_female"] = (scot_metadata["sex"] == "Female").astype(float)
scot_metadata["is_vaccinated"] = (scot_metadata["vacc_dose_number"] > 0).astype(float)

scot_metadata.to_parquet("../data/processed/scotland_sequence_metadata.parquet")

nextclade_res.to_parquet("../data/processed/scotland_nextclade_result.parquet")

### Consolidating Long-format Multiscale Clustering Output with Metadata and Contextual Data

In [2]:
scot_clustering = pd.read_parquet("../data/processed/cog_all_scotland_aligned_cluster_assignments_long.parquet")
scot_metadata = sc.load("../data/processed/scotland_sequence_metadata.obj")
dz_scot_vacc = sc.load("../data/processed/scotland_datazone_vaccinations.obj")
dz_scot_simd = sc.load("../data/processed/scotland_datazone_simd_data.obj")
dz_scot_testing = sc.load("../data/processed/scotland_testing.obj")
dz_scot_testing = dz_scot_testing[
    (dz_scot_testing["collection_date"] >= scot_metadata["collection_date"].min()) &
    (dz_scot_testing["collection_date"] <= scot_metadata["collection_date"].max())
]

scot_metadata.drop(columns=["vaccination_date"], inplace=True) # avoid confusion on merge

windows = build_windows(
    min_date=scot_metadata["collection_date"].min(),
    max_date=scot_metadata["collection_date"].max(),
    window_size= timedelta(weeks=3),
    step=timedelta(weeks=1),
)
assert len(windows) == scot_clustering["window_id"].nunique()

singletons = sc.ddict(list)
windows_info = sc.ddict(list)

for i, (start_date, end_date) in enumerate(windows, start=1):
    window = f"W{str(i).zfill(3)}"
    wdf = scot_metadata[(scot_metadata["collection_date"] >= start_date)
                        & (scot_metadata["collection_date"] < end_date)]
    tdf = dz_scot_testing[(dz_scot_testing["collection_date"] >= start_date)
                          & (dz_scot_testing["collection_date"] < end_date)]

    windows_info["window_id"].append(window)
    windows_info["wn_no_sequences"].append(wdf["sequence_id"].nunique())
    windows_info["wn_positive_tests"].append(tdf["dz_positive_tests"].sum())
    windows_info["wn_prop_sequenced"].append(wdf["sequence_id"].nunique() / tdf["dz_positive_tests"].sum())
    windows_info["wn_no_seq_dzs"] = wdf["datazone"].nunique()
    windows_info["wn_no_pos_test_dzs"] = tdf[tdf["dz_positive_tests"]>0]["datazone"].nunique()

    for lin, gdf in wdf.groupby("pango_lineage", sort=False):
        no_seqs = gdf["sequence_id"].nunique()
        if no_seqs == 1:
            for res in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]:
                singletons["sequence_id"].append(gdf["sequence_id"].iloc[0])
                singletons["window_id"].append(window)
                singletons["resolution"].append(res)
                singletons["cluster_id"].append(f"{window}|{lin}|R{res}|C0")

singletons = pd.DataFrame(singletons)
windows_info = pd.DataFrame(windows_info)

scot_clustering_x = pd.concat([scot_clustering, singletons], ignore_index=True)
assert len(scot_clustering_x) == (len(scot_clustering) + len(singletons))

full_metadata = scot_metadata.merge(dz_scot_simd, on='datazone', how='inner')
# Corrected final merge
scot_clustering_analysis_dataset = (
    scot_clustering_x
    .merge(full_metadata, on='sequence_id', how='inner')
    .merge(windows_info, on='window_id', how='inner')
    .merge(dz_scot_testing, on=['collection_date', 'datazone'], how='left')
    .merge(dz_scot_vacc, left_on=["collection_date", "datazone"], right_on=["vaccination_date", "datazone"], how="left")
)

scot_clustering_analysis_dataset["window_idx"] = (
            scot_clustering_analysis_dataset["window_id"].astype(str).str.extract(r"(\d+)", expand=False).astype(int)
)

windows = (
    scot_clustering_analysis_dataset.groupby("window_idx")
    .agg(wn_start_date=("collection_date", "min"),
         wn_end_date=("collection_date", "max"))
    .reset_index()
)

windows['wn_mid_date'] = windows['wn_start_date'] + (windows['wn_end_date'] - windows['wn_start_date']) / 2

scot_clustering_analysis_dataset = scot_clustering_analysis_dataset.merge(windows, on="window_idx", how="left")

scot_clustering_analysis_dataset["dz_prop_vaccinated"] = (
    scot_clustering_analysis_dataset["dz_total_vaccinated"] / scot_clustering_analysis_dataset["dz_population"]
)

scot_clustering_analysis_dataset.drop(columns=['vaccination_date'], inplace=True)

unique_ids = scot_clustering_analysis_dataset['patient_id'].unique()
patient_map: dict = {old_id: f"P{i:06d}" for i, old_id in enumerate(unique_ids, start=1)}
scot_clustering_analysis_dataset['patient_id'] = scot_clustering_analysis_dataset['patient_id'].map(patient_map)

print(f"Final dataset shape: {scot_clustering_analysis_dataset.shape}")

Final dataset shape: (7395680, 52)


In [4]:
sorted_columns = [
     # === Window-Level Info (wn_) ===
    'window_idx',
    'window_id',
    'wn_start_date',
    'wn_mid_date',
    'wn_end_date',
    'wn_no_sequences',
    'wn_positive_tests',
    'wn_prop_sequenced',
    'wn_no_seq_dzs',
    'wn_no_pos_test_dzs',

    # === Core Identifiers & Clustering Info ===
    'sequence_id',
    'patient_id',
    'resolution',
    'cluster_id',

    # === Core Metadata: Time, Place, and Person ===
    'collection_date',
    'datazone',
    'dz_x',  # datazone centroid x coordinates
    'dz_y',  # datazone centroid y coordinates
    'sex',
    'is_female',
    'age_band',
    'age_midpoint',
    'is_vaccinated',
    'vacc_dose_number',

    # === Genomic Information ===
    'pango_lineage',
    'clade',
    'who_voc',
    'nextclade_qc',

    # === Datazone Demographics & SIMD (dz_) ===
    'dz_population',
    'dz_working_age_population',
    'dz_simd_rank',
    'dz_simd_quintile',
    'dz_simd_decile',
    'dz_simd_vigintile',
    'dz_simd_income_rank',
    'dz_simd_employment_rank',
    'dz_simd_education_rank',
    'dz_simd_health_rank',
    'dz_simd_access_rank',
    'dz_simd_crime_rank',
    'dz_simd_housing_rank',

    # === Datazone Daily Testing & Vaccination Stats (dz_) ===
    'dz_total_tests',
    'dz_positive_tests',
    'dz_negative_tests',
    'dz_total_vaccinated',
    'dz_prop_vaccinated'
]

scot_clustering_analysis_dataset_sorted = scot_clustering_analysis_dataset[sorted_columns].copy()

scot_clustering_analysis_dataset_sorted.reset_index(drop=True, inplace=True)

print(f"Final dataset shape: {scot_clustering_analysis_dataset_sorted.shape}")
print(f"Columns:\n{list(scot_clustering_analysis_dataset_sorted.columns)}")

Final dataset shape: (7395680, 46)
Columns:
['window_idx', 'window_id', 'wn_start_date', 'wn_mid_date', 'wn_end_date', 'wn_no_sequences', 'wn_positive_tests', 'wn_prop_sequenced', 'wn_no_seq_dzs', 'wn_no_pos_test_dzs', 'sequence_id', 'patient_id', 'resolution', 'cluster_id', 'collection_date', 'datazone', 'dz_x', 'dz_y', 'sex', 'is_female', 'age_band', 'age_midpoint', 'is_vaccinated', 'vacc_dose_number', 'pango_lineage', 'clade', 'who_voc', 'nextclade_qc', 'dz_population', 'dz_working_age_population', 'dz_simd_rank', 'dz_simd_quintile', 'dz_simd_decile', 'dz_simd_vigintile', 'dz_simd_income_rank', 'dz_simd_employment_rank', 'dz_simd_education_rank', 'dz_simd_health_rank', 'dz_simd_access_rank', 'dz_simd_crime_rank', 'dz_simd_housing_rank', 'dz_total_tests', 'dz_positive_tests', 'dz_negative_tests', 'dz_total_vaccinated', 'dz_prop_vaccinated']


In [5]:
sc.save(
    "../data/processed/scotland_clustering_analysis_dataset.obj",
    scot_clustering_analysis_dataset_sorted, compression="zstd"
)

scot_clustering_analysis_dataset_sorted.to_parquet(
    "../data/processed/scotland_clustering_analysis_dataset.parquet",
    index=False, compression="zstd"
)

In [10]:
display(scot_clustering_analysis_dataset_sorted.head().T)

,0,1,2,3,4
window_idx,1,1,1,1,1
window_id,W001,W001,W001,W001,W001
wn_start_date,2020-07-04 00:00:00,2020-07-04 00:00:00,2020-07-04 00:00:00,2020-07-04 00:00:00,2020-07-04 00:00:00
wn_mid_date,2020-07-14 00:00:00,2020-07-14 00:00:00,2020-07-14 00:00:00,2020-07-14 00:00:00,2020-07-14 00:00:00
wn_end_date,2020-07-24 00:00:00,2020-07-24 00:00:00,2020-07-24 00:00:00,2020-07-24 00:00:00,2020-07-24 00:00:00
wn_no_sequences,63,63,63,63,63
wn_positive_tests,342.0,342.0,342.0,342.0,342.0
wn_prop_sequenced,0.184211,0.184211,0.184211,0.184211,0.184211
wn_no_seq_dzs,776,776,776,776,776
wn_no_pos_test_dzs,2798,2798,2798,2798,2798



### [PANGO Lineage](https://academic.oup.com/ve/article/7/2/veab064/6315289)

**Purpose:** To provide a finer-grained, dynamic system for naming and tracking specific transmission lineages of a virus, based on epidemiological evidence.

**Granularity:** Fine-scaled and hierarchical. PANGO lineages are created to define distinct phylogenetic clusters, often corresponding to specific transmission events or introductions into a geographic area.

**Naming convention:** Hierarchical letters and numbers, such as "B.1.1.7." Descendent lineages get an additional number, like "BA.2" and "BA.5" within the Omicron lineage.

**Relationship to Clades:** PANGO is a much more detailed system than Nextstrain. A Nextstrain clade often encompasses multiple PANGO lineages.